One sequence input

In [11]:
# scripts/example.py
# -*- coding: utf-8 -*-
"""End-to-end example for Directed Evolution in Codon Space.

Runs three steps on a single parental CDS:
  1. Evo score
  2. Single-point synonymous mutation scan
  3. Directed evolution (greedy walk, 10% mutation cap)

Model weights, tokenizer, and config are pulled from Hugging Face
automatically the first time you run this.
"""

from evolve import Evolver
from codon_utils import print_optimization_report, print_mutation_scan


# -------------------------------------------------------------------
# Inputs
# -------------------------------------------------------------------

HOST_TOKEN_TYPE = 345   # C. griseus (CHO). See species_token_type.py for the full list.

# Parental CDS - replace with your own full-length coding sequence.
PARENTAL_CDS = (
    "ATCCAGCTGACCCAGAGCCCCAGCAGCCTGAGCGCCAGCGTGGGCGACCGGGTG"
)


def section(title):
    """Print a visual section divider."""
    bar = "=" * 72
    print(f"\n{bar}\n{title}\n{bar}")


def main():
    # ---------------------------------------------------------------
    # Load SynCodonLM v2 from Hugging Face
    # ---------------------------------------------------------------
    section("Loading SynCodonLM v2 (Hugging Face)")
    evolver = Evolver()   # downloads + caches on first use

    # ---------------------------------------------------------------
    # 1. Evo scoring
    # ---------------------------------------------------------------
    section("1. Evo scoring")
    score = evolver.evo_score(PARENTAL_CDS, token_type_id=HOST_TOKEN_TYPE)
    print(f"Length (codons)                : {len(PARENTAL_CDS) // 3}")
    print(f"Evo score (synonymous-constrained) : {score.synonymous_constrained:+.4f} nats")
    print(f"Evo score (unconstrained)          : {score.unconstrained:+.4f} nats")

    # ---------------------------------------------------------------
    # 2. Single-point synonymous mutation scan
    # ---------------------------------------------------------------
    section("2. Preferred single-point synonymous mutations")
    scan = evolver.scan_mutations(
        PARENTAL_CDS,
        token_type_id=HOST_TOKEN_TYPE,
        min_delta_nats=0.0,
        sort=True,
    )

    if not scan:
        print("No positions where the model prefers a different synonym.")
    else:
        print(f"{len(scan)} preferred single-point mutations found.\n")
        print_mutation_scan(scan)

        top_n = min(5, len(scan))
        print(f"\nTop-{top_n} single-point recipe:")
        for rank, m in enumerate(scan[:top_n], start=1):
            print(f"  {rank}. pos {m.pos_1based:>4}  ({m.aa})  "
                  f"{m.from_codon} -> {m.to_codon}   "
                  f"delta = {m.delta_nats:+.4f} nats")

    # ---------------------------------------------------------------
    # 3. Directed-evolution walk
    # ---------------------------------------------------------------
    section("3. Directed evolution (10% mutation cap)")
    results = evolver.optimize(
        [PARENTAL_CDS],
        token_type_id=HOST_TOKEN_TYPE,
        max_change_fraction=0.10,
    )
    result = results[0] #results is a list, as it can take numerous sequences in one pass
    print_optimization_report(results)
    print(f"Here is your optimized DNA: {result.optimized_dna}")


if __name__ == "__main__":
    main()



Loading SynCodonLM v2 (Hugging Face)
[Evolver] Loading SynCodonLM V2 from Hugging Face: jheuschkel/SynCodonLM-V2
[Evolver]   device: cuda   dtype: torch.float16
[Evolver] Loaded. Parameters: 102.6M   token_type_ids supported: True

1. Evo scoring
Length (codons)                : 18
Evo score (synonymous-constrained) : -0.5608 nats
Evo score (unconstrained)          : -3.2354 nats

2. Preferred single-point synonymous mutations
2 preferred single-point mutations found.

   pos  aa  from    to    delta (nats)    odds   runner-up
    11   S   AGC -> TCG         0.5029   1.654   TCT
    17   R   CGG -> AGA         0.3701   1.448   CGG

Top-2 single-point recipe:
  1. pos   11  (S)  AGC -> TCG   delta = +0.5029 nats
  2. pos   17  (R)  CGG -> AGA   delta = +0.3701 nats

3. Directed evolution (10% mutation cap)
 Sequence #0  (L = 18 codons) 
Original  evo score (syn)      : -0.560832 nats
Optimized evo score (syn)      : -0.550030 nats
Delta evo score (syn)          : +0.010802 nats (odds x

Two+ sequences input 

In [14]:
# scripts/example.py
# -*- coding: utf-8 -*-

"""End-to-end example for Directed Evolution in Codon Space.

Runs three steps for each parental CDS:
  1. Evo score
  2. Single-point synonymous mutation scan
  3. Directed evolution with a 10% mutation cap

Model weights, tokenizer, and config are pulled from Hugging Face
automatically the first time the script runs.
"""

from evolve import Evolver
from codon_utils import print_optimization_report, print_mutation_scan


# -------------------------------------------------------------------
# Inputs
# -------------------------------------------------------------------

HOST_TOKEN_TYPE = 345  # C. griseus (CHO)

PARENTAL_CDS = [
    "ATCCAGCTGACCCAGAGCCCCAGCAGCCTGAGCGCCAGCGTGGGCGACCGGGTG",
    "TCCAGCTGACCCAGAGCCCCAGCAGCCTGAGCGCCAGCGTGGGCGACCGGG",
]


def section(title):
    """Print a visual section divider."""
    bar = "=" * 72
    print(f"\n{bar}\n{title}\n{bar}")


def validate_sequences(sequences):
    """Validate that each sequence is a nonempty, in-frame DNA CDS."""

    cleaned_sequences = []

    for sequence_number, sequence in enumerate(sequences, start=1):
        cleaned = "".join(sequence.split()).upper().replace("U", "T")

        if not cleaned:
            raise ValueError(
                f"Sequence {sequence_number} is empty."
            )

        invalid_characters = sorted(set(cleaned) - set("ACGT"))

        if invalid_characters:
            raise ValueError(
                f"Sequence {sequence_number} contains invalid DNA characters: "
                f"{invalid_characters}"
            )

        if len(cleaned) % 3 != 0:
            raise ValueError(
                f"Sequence {sequence_number} has {len(cleaned)} nucleotides, "
                "which is not divisible by 3. Verify the CDS frame."
            )

        cleaned_sequences.append(cleaned)

    return cleaned_sequences


def main():
    # ---------------------------------------------------------------
    # Validate input sequences
    # ---------------------------------------------------------------
    sequences = validate_sequences(PARENTAL_CDS)

    # ---------------------------------------------------------------
    # Load SynCodonLM v2
    # ---------------------------------------------------------------
    section("Loading SynCodonLM v2 from Hugging Face")

    evolver = Evolver()

    # ---------------------------------------------------------------
    # 1. Evo scoring
    # ---------------------------------------------------------------
    section("1. Evo scoring")

    for sequence_number, dna in enumerate(sequences, start=1):
        score = evolver.evo_score(
            dna,
            token_type_id=HOST_TOKEN_TYPE,
        )

        print(f"\nSequence {sequence_number}")
        print(f"Length (nucleotides)                : {len(dna)}")
        print(f"Length (codons)                     : {len(dna) // 3}")
        print(
            "Evo score (synonymous-constrained) : "
            f"{score.synonymous_constrained:+.4f} nats"
        )
        print(
            "Evo score (unconstrained)          : "
            f"{score.unconstrained:+.4f} nats"
        )

    # ---------------------------------------------------------------
    # 2. Single-point synonymous mutation scans
    # ---------------------------------------------------------------
    section("2. Preferred single-point synonymous mutations")

    for sequence_number, dna in enumerate(sequences, start=1):
        print(f"\nSequence {sequence_number}")
        print("-" * 72)

        scan = evolver.scan_mutations(
            dna,
            token_type_id=HOST_TOKEN_TYPE,
            min_delta_nats=0.0,
            sort=True,
        )

        if not scan:
            print(
                "No positions where the model prefers a different synonym."
            )
            continue

        print(f"{len(scan)} preferred single-point mutations found.\n")
        print_mutation_scan(scan)

        top_n = min(5, len(scan))

        print(f"\nTop-{top_n} single-point recipe:")

        for rank, mutation in enumerate(scan[:top_n], start=1):
            print(
                f"  {rank}. "
                f"pos {mutation.pos_1based:>4}  "
                f"({mutation.aa})  "
                f"{mutation.from_codon} -> {mutation.to_codon}   "
                f"delta = {mutation.delta_nats:+.4f} nats"
            )

    # ---------------------------------------------------------------
    # 3. Directed-evolution walks
    # ---------------------------------------------------------------
    section("3. Directed evolution with a 10% mutation cap")

    # PARENTAL_CDS is already a list. Do not wrap it in another list.
    results = evolver.optimize(
        sequences,
        token_type_id=HOST_TOKEN_TYPE,
        max_change_fraction=0.10,
    )

    print_optimization_report(results)

    # One OptimizationResult is returned for each input CDS.
    section("Optimized DNA sequences")

    for result in results:
        sequence_number = result.input_index + 1

        print(f"\nSequence {sequence_number}")
        print("-" * 72)

        print("Original DNA:")
        print(result.original_dna)

        print("\nOptimized DNA:")
        print(result.optimized_dna)

        print("\nSummary:")
        print(f"Length                     : {result.length_codons} codons")
        print(f"Positions changed          : {result.positions_changed}")
        print(
            f"Original constrained score : "
            f"{result.original_evo_score:+.4f} nats"
        )
        print(
            f"Optimized constrained score: "
            f"{result.optimized_evo_score:+.4f} nats"
        )
        print(
            f"Change in score            : "
            f"{result.delta_evo_score:+.4f} nats"
        )
        print(f"Stop reason                : {result.stop_reason}")


if __name__ == "__main__":
    main()


Loading SynCodonLM v2 from Hugging Face
[Evolver] Loading SynCodonLM V2 from Hugging Face: jheuschkel/SynCodonLM-V2
[Evolver]   device: cuda   dtype: torch.float16
[Evolver] Loaded. Parameters: 102.6M   token_type_ids supported: True

1. Evo scoring

Sequence 1
Length (nucleotides)                : 54
Length (codons)                     : 18
Evo score (synonymous-constrained) : -0.5608 nats
Evo score (unconstrained)          : -3.2354 nats

Sequence 2
Length (nucleotides)                : 51
Length (codons)                     : 17
Evo score (synonymous-constrained) : -1.1839 nats
Evo score (unconstrained)          : -4.1531 nats

2. Preferred single-point synonymous mutations

Sequence 1
------------------------------------------------------------------------
2 preferred single-point mutations found.

   pos  aa  from    to    delta (nats)    odds   runner-up
    11   S   AGC -> TCG         0.5029   1.654   TCT
    17   R   CGG -> AGA         0.3701   1.448   CGG

Top-2 single-point 